In [16]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [30]:
# Following https://github.com/MrNeRF/LichtFeld-Studio/pull/1314, in order to create a Segment+Ignore mask, we need to combine the car masks (ignore) with the sky masks (segment).
# Gray values:
# 0 -> 128 elements to ignore
# 128 -> 250 elements to segment
# 250 -> 255 elements to keep

resolution = (7130, 7130)  # Resolution of the masks

# Get all JPG mask files in the masks_car and masks_sky directories
masks_car_dir = "RS_export/masks_car"
masks_sky_dir = "RS_export/masks_sky"
masks_combined_dir = "RS_export/masks"
os.makedirs(masks_combined_dir, exist_ok=True)
masks_car_files = [f for f in os.listdir(masks_car_dir) if f.endswith(".jpg")]
masks_sky_files = [f for f in os.listdir(masks_sky_dir) if f.endswith(".jpg")]
# Combined masks as union of masks_car and masks_sky
masks_combined_files = list(set(masks_car_files) | set(masks_sky_files))

# Get image files
# masks_car = [cv2.imread(os.path.join(masks_car_dir, f), cv2.IMREAD_GRAYSCALE) for f in tqdm(masks_car_files)]
# masks_sky = [cv2.imread(os.path.join(masks_sky_dir, f), cv2.IMREAD_GRAYSCALE) for f in tqdm(masks_sky_files)]

# ^^^ This is a bad idea, don't load them into memory all at once

In [32]:
# In car masks, car parts are white, rest is black. In sky masks, sky parts are white, rest is black.
# We want masks_car white -> 0, black -> 255
# We want masks_sky white -> 150, black -> 255
# After this transformation, combine them such that 255 is only kept if both masks are 255, otherwise take the lower value (0 or 150, the regions won't overlap but check in case).
for mask_file in tqdm(masks_combined_files):
	mask_car_path = os.path.join(masks_car_dir, mask_file)
	mask_sky_path = os.path.join(masks_sky_dir, mask_file)
	# Load masks if they exist, otherwise create a black mask
	if os.path.exists(mask_car_path):
		mask_car = cv2.imread(mask_car_path, cv2.IMREAD_GRAYSCALE)
		# Transform car mask: white -> 0, black -> 255
		mask_car = np.where(mask_car == 255, 0, 255).astype(np.uint8)
	else:
		mask_car = np.full(resolution, 255, dtype=np.uint8)  # Black mask

	if os.path.exists(mask_sky_path):
		mask_sky = cv2.imread(mask_sky_path, cv2.IMREAD_GRAYSCALE)
		# Transform sky mask: white -> 150, black -> 255
		mask_sky = np.where(mask_sky == 255, 150, 255).astype(np.uint8)
	else:
		mask_sky = np.full(resolution, 255, dtype=np.uint8)  # Black mask

	# Combine the masks
	mask_combined = np.minimum(mask_car, mask_sky)

	# Save the combined mask IF the entire masks is NOT white
	if np.all(mask_combined == 255):
		print(f"Skipping {mask_file} because the combined mask is all white.")
		continue
	cv2.imwrite(os.path.join(masks_combined_dir, mask_file), mask_combined)

Skipping frame_000170.jpg because the combined mask is all white.


Skipping frame_000317.jpg because the combined mask is all white.


Skipping frame_000161.jpg because the combined mask is all white.


Skipping frame_000335.jpg because the combined mask is all white.


Skipping frame_000310.jpg because the combined mask is all white.


...

Skipping frame_000162.jpg because the combined mask is all white.


Skipping frame_000343.jpg because the combined mask is all white.


Skipping frame_000333.jpg because the combined mask is all white.


Skipping frame_000320.jpg because the combined mask is all white.


100%|██████████| 252/252 [02:15<00:00,  1.86it/s]
